# Lesson 1: Accessing Claude with the API
## 1.1 Making a request

In [2]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Make a request
message = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [10]:
message.content[0].text

'Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing quantum computers to process certain types of problems exponentially faster than classical computers.'

## 1.2 Multi-Turn Conversations

- Anthropic API and Claude do not store any messages
- To have a conversation, you need to:
    1) manually maintain a list of messages in your code
    2) provide that list of messages with each follow up request

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
    model=model,
    max_tokens=200,
    messages=messages
    )
    return message.content[0].text


In [15]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# Pass the list of messages into 'chat' to get an answer 
result = chat(messages)

# Take the answer and add it as an assistant message into our list 
add_assistant_message(messages, result)

# Add in the user's follow up question
add_user_message(messages, "Write another sentence")

# Call chat again with the list of messages to get the final answer
result = chat(messages)
result

'Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, enabling quantum computers to explore many possible solutions to a problem in parallel.'

## 1.3 Chat Exercise

Make a chatbot using the three helper functions we just put together above. 

1. Prompt the user to enter some input using the "built-in" input function
2. Add it to a list of messages
3. Call the API
4. Add generated text to the list of messages 
5. Print the generated text
6. Repeat from #1

Chat Bot Exercise (My Work)

In [5]:
def chatbot():
    messages = []
    while True:
        user_input = input("Prompt: ")
        add_user_message(messages, user_input)
        result = chat(messages)
        print(result)
        add_assistant_message(messages, result)
        
        while True:
            choice = input("Do you wish to continue? Y/N ").lower().strip()
            if choice == "y":
                break
            elif choice == "n":
                print("Goodbye!")
                return
            else:
                print("Please type Y/N only") 
        
chatbot()

Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, enabling certain problems to be solved exponentially faster.
Unlike classical bits that exist as either 0 or 1, quantum bits (qubits) can exist in superposition—simultaneously representing both states—until measured, allowing quantum computers to explore multiple solutions in parallel.
Goodbye!


Chat Bot Exercise Solution

In [ ]:
# Make an intial list of messages
messages = []

# Use a while True loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    print(">", user_input)
    
    # Add user input to the list of messages
    add_user_message(messages, user_input)
    
    # Call Claude with the 'chat' function
    answer = chat(messages)
    
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    
    # Print the generated text
    print("---")
    print(answer)
    print("---")

## 1.4 System Prompting

- system prompts provide Claude guidance on how to respond 
- Claude will try to respond in the same way someine in the specified role would respond 
- Helps keep Claude on task
- Used to customize the tone and style of response

In [14]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model":model,
        "max_tokens":200,
        "messages":messages,
    }
    
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [12]:
messages = []

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
result = chat(messages, system=system)

result

"Great question! Let's work through this step by step.\n\nThe goal is to get x by itself on one side of the equation.\n\n**Step 1:** Look at what's with the x on the left side. You have 5x + 3. \n\nWhat do you need to remove first to start isolating x — the +3 or the ×5?\n\n(Hint: Think about the order of operations in reverse!)"

## 1.5 System Prompts Exercise

In [15]:
messages = []

system = """
You are an expert Python programmer.
Use pythonic methods to write functions.
Do not give multiple examples, give the best one. 
Keep your code concise.
"""

add_user_message(messages, "Write a Python function that checks a string for duplicate characters.")
result = chat(messages, system=system)

result

'# Python Function to Check for Duplicate Characters\n\nHere are several approaches, from simple to comprehensive:\n\n## 1. **Simple Boolean Check** (Most Pythonic)\n```python\ndef has_duplicates(s: str) -> bool:\n    """Check if a string has duplicate characters."""\n    return len(s) != len(set(s))\n```\n\n**Example:**\n```python\nprint(has_duplicates("hello"))      # True (l appears twice)\nprint(has_duplicates("python"))     # False (all unique)\nprint(has_duplicates("aabbcc"))     # True\n```\n\n---\n\n## 2. **Find & Return Duplicate Characters**\n```python\ndef get_duplicates(s: str) -> set:\n    """Return a set of duplicate characters in a string."""\n    seen = set()\n    duplicates = set()\n    '

## 1.6 Temperature

- temperature parameter changes how likely each token is to be selected
- Need consistent, factual responses? Use low temperature
- Want creative brainstorming? Dial up the temperature
- Somewhere in between? Medium temperatures work well for most general tasks

In [16]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 200,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

Testing temperature effects

In [17]:
# Low temperature - more predictable
messages = []

add_user_message(messages, "Generate a one sentence money-making idea in Singapore.")

answer = chat(messages, temperature=0.0)

answer

'Start a virtual assistant service for busy professionals and small business owners in Singapore, offering administrative support, scheduling, and customer service tasks remotely at competitive rates.'

In [19]:
# High temperature - more creative  
messages = []

add_user_message(messages, "Generate a one sentence money-making idea in Singapore.")

answer = chat(messages, temperature=1.0)

answer

'Start a freelance content creation service for e-commerce businesses in Southeast Asia, helping them produce product photography, videos, and descriptions tailored for Lazada and Shopee.'

## 1.7 Response Streaming

When you enable streaming, Claude sends back several types of events:

- MessageStart - A new message is being sent
- ContentBlockStart - Start of a new block containing text, tool use, or other content
- ContentBlockDelta - Chunks of the actual generated text
- ContentBlockStop - The current content block has been completed
- MessageDelta - The current message is complete
- MessageStop - End of information about the current message

In [20]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model":model,
        "max_tokens":200,
        "messages":messages,
    }
    
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [24]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=200,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()
    

ParsedMessage(id='msg_011CdboX2KdKVimMCVREVLo5', container=None, content=[ParsedTextBlock(citations=None, text='# Fake Database Description\n\nThe "CloudDream" database is a distributed NoSQL system that uses quantum-inspired algorithms to predict user behavior patterns and automatically reorganizes data across nodes based on emotional relevance rather than traditional indexing.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=51, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

## 1.8 Structured Data

In [28]:
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":200,
        "messages":messages,
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages)

- we only want the json content and not the additional fluff around like the json or markdown backticks

In [30]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [31]:
import json

json.loads(text.strip())

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

## 1.9 Structured Data Exercise

- use message prefilling and stop sequences only to get three different commands in a single response
- there shouldn't be any comments or explanation
- message prefilling is not limited to just characters like ```

In [41]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(
    messages, 
    "Here are all three commands in a single block without any comments:\n ```bash"
)

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws dynamodb list-tables'